# Engine: JWST Spectral Pixel

**Claim:** Each JWST pixel is an octonion element. 8 NIRCam filters = 8 octonion components e₀…e₇. One 𝕆 element per sky pixel.

```
JWST NIRCam filters → octonion address:
  F090W : 900 nm  → e₀
  F115W : 1150 nm → e₁
  F150W : 1500 nm → e₂
  F200W : 2000 nm → e₃
  F277W : 2770 nm → e₄
  F356W : 3560 nm → e₅
  F410M : 4100 nm → e₆
  F444W : 4440 nm → e₇
```

**File:** `ValaQuenta/modules/jwst/maths.py`

The Cayley-Dickson tower provides the natural address space for spectral data. No dimension reduction. No information loss. 8 filters = 8 dimensions = 𝕆.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../..'))
from ValaQuenta.modules.jwst import maths as jw
import math
print('jwst module loaded')
print(f'Filters: {jw.FILTER_NAMES}')
print(f'Wavelength range: {jw.LAMBDA_MIN}–{jw.LAMBDA_MAX} nm')

## Filter → Wavelength → Radial Coordinate

In [ ]:
print(f'{'Filter':8s}  {'λ (nm)':8s}  {'r (radial)':10s}  {'Octonion'}')
for i, (fname, lam) in enumerate(jw.JWST_FILTERS.items()):
    r = jw.wavelength_to_radial(lam)
    print(f'{fname:8s}  {lam:8.0f}  {r:10.6f}  e{i}')

## Pixel → Octonion Element

In [ ]:
# Synthetic pixel (8 filter intensities)
pixel_intensities = [0.8, 0.6, 0.5, 0.4, 0.35, 0.3, 0.25, 0.2]

octonion_elem = jw.pixel_to_octonion(pixel_intensities)
print('Pixel intensities → Octonion element:')
for i, (comp, intensity) in enumerate(zip(octonion_elem.components, pixel_intensities)):
    print(f'  e{i} (F{["090W","115W","150W","200W","277W","356W","410M","444W"][i]}): intensity={intensity:.2f}  component={comp:.6f}')
print(f'\nOctonion norm: {octonion_elem.norm:.6f}')
print(f'(Norm = total spectral energy of the pixel)')

## Cayley-Dickson Address — Which Strata?

In [ ]:
# The norm determines which CD stratum the pixel lives in
norm = octonion_elem.norm
D_STAR = 0.24600
OMEGA_ZS = 0.5671432904097838

print(f'Pixel norm: {norm:.6f}')
print()
if norm < D_STAR:
    stratum = 'Below d* — conformal boundary. Zero-divisor territory.'
elif norm < OMEGA_ZS:
    stratum = 'Between d* and OMEGA_ZS — the critical band.'
elif norm < 1.0:
    stratum = 'Between OMEGA_ZS and 1 — main sedenion body.'
else:
    stratum = 'Above 1 — exterior. Outward expansion regime.'
print(f'Stratum: {stratum}')

# Show multiple pixel types
test_pixels = [
    ('faint galaxy',  [0.05, 0.04, 0.03, 0.02, 0.01, 0.008, 0.006, 0.004]),
    ('bright star',   [0.99, 0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.65]),
    ('edge-on disk',  [0.3, 0.6, 0.9, 0.7, 0.5, 0.4, 0.3, 0.2]),
]
print()
print(f'{'Object':15s}  Norm     Stratum')
for name, intens in test_pixels:
    oct_e = jw.pixel_to_octonion(intens)
    n = oct_e.norm
    s = 'ZD territory' if n < D_STAR else ('critical band' if n < OMEGA_ZS else 'main body')
    print(f'{name:15s}  {n:.6f}  {s}')

## Redshift → Radial Coordinate Shift

In [ ]:
# Cosmological redshift shifts the filter-to-octonion mapping
print('Redshift z shifts the wavelength → octonion mapping:')
for z in [0.0, 0.5, 1.0, 2.0, 5.0]:
    shift = jw.redshift_to_radial_shift(z)
    print(f'  z = {z:.1f}:  Δr = {shift:.6f}')
print()
print('At z=OMEGA_ZS≈0.567: observer sits at the BAO attractor.')